# 🌐 Aula 13 — A distribuição normal e a velocidade de um site

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte do **time de dados de uma loja online**. Uma das métricas mais acompanhadas pelo time é o **tempo de carregamento** das páginas: páginas lentas fazem o cliente desistir da compra. Pelo histórico, esse tempo tem a forma de sino da **distribuição normal**, a ferramenta da Aula 13.

> ⚠️ Os números usados aqui são **fictícios**, escolhidos só para ilustrar os modelos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**;
- **`scipy.stats.norm`**: é a **tabela da normal padrão** da aula, só que dentro do computador. Duas funções dela aparecem neste notebook:
  - `norm.cdf(z)` dá $\Phi(z) = P(Z \le z)$, a área à **esquerda** de $z$ (o que você procura na tabela a partir de $z$);
  - `norm.ppf(p)` faz o caminho **contrário**: dada a área $p$, devolve o $z$ com $P(Z \le z) = p$ (o que você faz quando procura a área no meio da tabela e lê o $z$ na margem).

E a ferramenta central da aula, a **padronização**: se $X \sim N(\mu, \sigma^2)$, então
$$Z = \frac{X - \mu}{\sigma} \sim N(0, 1).$$

Criamos também o **sorteador** (`rng`) com a **semente 42**, que faz as simulações saírem sempre iguais.

**Lembrete:** **empírico** = o que sai da simulação; **teórico** = o que a fórmula (ou a tabela) diz.

In [ ]:
import numpy as np                    # calculadora de números e sorteios
import matplotlib.pyplot as plt       # gráficos
from scipy.stats import norm          # a "tabela da normal padrão"

rng = np.random.default_rng(42)       # sorteador, semente 42
print("Tudo pronto!")
print("Teste da tabela: P(Z <= 1,00) =", round(norm.cdf(1.00), 4))   # na tabela: 0,8413

## 1. Quantas páginas carregam rápido?

### A situação
O tempo de carregamento da página de produto, em segundos, segue uma distribuição **normal** com média $\mu = 2{,}0$ s e desvio padrão $\sigma = 0{,}5$ s.

### A pergunta
Seja $X$ = tempo de carregamento de uma página. Qual a fração das páginas que carrega:
1. em **até 2,5 s**?
2. em **mais de 3 s** (o limite em que muitos clientes desistem)?
3. **entre 1,5 s e 2,5 s**?

### A ferramenta da aula
$X \sim N(2{,}0;\ 0{,}5^2)$. Padronizamos cada limite com $z = \dfrac{x - 2{,}0}{0{,}5}$ e consultamos a tabela.

### A resposta pela conta
1. $z = \frac{2{,}5 - 2{,}0}{0{,}5} = 1$, então $P(X \le 2{,}5) = \Phi(1) \approx 0{,}8413$;
2. $z = \frac{3 - 2{,}0}{0{,}5} = 2$, então $P(X > 3) = 1 - \Phi(2) \approx 1 - 0{,}9772 = 0{,}0228$;
3. $z$ vai de $-1$ a $1$: $P(1{,}5 \le X \le 2{,}5) = \Phi(1) - \Phi(-1) \approx 0{,}8413 - 0{,}1587 = 0{,}6827$.

### O que o código faz
1. escreve a padronização como uma função do Python;
2. calcula as três respostas com a "tabela" `norm.cdf`;
3. **simula o carregamento de 100.000 páginas** (`rng.normal` sorteia valores de uma normal) e compara.

In [ ]:
mu, sigma = 2.0, 0.5                                   # média e desvio padrão, em segundos


def padroniza(x):
    # transforma um tempo x no valor z da normal padrão
    return (x - mu) / sigma


p1 = norm.cdf(padroniza(2.5))                          # P(X <= 2,5)
p2 = 1 - norm.cdf(padroniza(3.0))                      # P(X > 3)
p3 = norm.cdf(padroniza(2.5)) - norm.cdf(padroniza(1.5))   # P(1,5 <= X <= 2,5)

tempos = rng.normal(mu, sigma, size=100_000)           # 100.000 páginas simuladas

print(f"P(X <= 2,5)        z = {padroniza(2.5):.0f}    teórico: {p1:.4f}   empírico: {(tempos <= 2.5).mean():.4f}")
print(f"P(X > 3)           z = {padroniza(3.0):.0f}    teórico: {p2:.4f}   empírico: {(tempos > 3).mean():.4f}")
print(f"P(1,5 <= X <= 2,5) z de -1 a 1  teórico: {p3:.4f}   empírico: {((tempos >= 1.5) & (tempos <= 2.5)).mean():.4f}")

**Como ler:**
- **84%** das páginas carregam em até 2,5 s;
- só cerca de **2,3%** passam de 3 s — algo como 23 páginas a cada 1.000;
- cerca de **68%** ficam a menos de um desvio padrão da média (entre 1,5 s e 2,5 s).

O gráfico abaixo mostra o histograma dos 100.000 tempos simulados com a curva da densidade normal por cima, e pinta de vermelho a área das páginas lentas (mais de 3 s). **A área pintada é a probabilidade** $P(X > 3)$.

In [ ]:
x = np.linspace(0, 4, 400)
densidade = norm.pdf(x, mu, sigma)                     # a curva em forma de sino

plt.hist(tempos, bins=60, density=True, color="lightsteelblue", edgecolor="gray")
plt.plot(x, densidade, color="steelblue", linewidth=2, label="densidade N(2,0; 0,5²)")
plt.fill_between(x, densidade, where=(x > 3), color="firebrick", label="P(X > 3) ≈ 2,3%")
plt.xlabel("Tempo de carregamento (segundos)")
plt.ylabel("Densidade")
plt.title("Tempo de carregamento das páginas")
plt.legend()
plt.show()

**Como ler:** o histograma da simulação "encaixa" na curva em forma de sino, simétrica em torno da média de 2 s. A área vermelha, à direita de 3 s, é bem fina: as páginas lentas são poucas, mas existem.

## 2. O caminho contrário: definindo a meta de velocidade (SLA)

### A situação e a pergunta
A diretoria quer publicar uma meta de serviço (um **SLA**) do tipo: "**95%** das páginas carregam em até $t$ segundos". **Qual o valor de $t$?** E se a meta for de **99%**?

### A ferramenta da aula
Agora a **área** é conhecida e o **valor** é que falta — o mesmo tipo de exercício da aula em que se procura $z$ tal que $P(Z \le z) = p$. Fazemos em dois passos:
1. na tabela, encontramos $z$ com $\Phi(z) = 0{,}95$;
2. desfazemos a padronização: $t = \mu + z \cdot \sigma$.

### A resposta pela conta
- 95%: $z \approx 1{,}645$, então $t = 2{,}0 + 1{,}645 \times 0{,}5 \approx 2{,}82$ s;
- 99%: $z \approx 2{,}326$, então $t = 2{,}0 + 2{,}326 \times 0{,}5 \approx 3{,}16$ s.

### O que o código faz
Usa `norm.ppf` (a tabela "ao contrário") para achar $z$ e depois desfaz a padronização. No fim, confere na simulação que fração das páginas ficou mesmo abaixo de cada $t$.

In [ ]:
for meta in (0.95, 0.99):
    z = norm.ppf(meta)                                 # z com P(Z <= z) = meta
    t = mu + z * sigma                                 # volta para segundos
    print(f"Meta de {meta:.0%}:  z = {z:.3f}   t = {t:.2f} s   (na simulação: {(tempos <= t).mean():.1%} abaixo de t)")

**Como ler:** para garantir que **95%** das páginas carreguem a tempo, o SLA deve prometer **2,82 s**; para **99%**, **3,16 s**. Repare como a meta fica mais cara à medida que se aproxima de 100%: de 95% para 99% são só 4 pontos percentuais, mas o tempo prometido sobe 0,34 s, porque estamos entrando na cauda da curva.

Esse valor $t$ tem nome em ciência de dados: é o **percentil** 95 (ou 99) do tempo de carregamento, uma das métricas mais usadas para monitorar sites e aplicativos.

## 3. Qual servidor é melhor?

### A situação
A empresa testa um **servidor novo (B)**. No servidor atual (A), o tempo é $N(2{,}0;\ 0{,}5^2)$. No servidor B, o tempo médio é **menor**, mas o tempo varia mais: $N(1{,}8;\ 0{,}9^2)$.

### A pergunta
Em qual dos dois servidores há **menos páginas lentas** (mais de 3 s)?

### A ferramenta da aula
Padronizamos o limite de 3 s em **cada** distribuição, com a média e o desvio padrão de cada servidor:
- A: $z = \frac{3 - 2{,}0}{0{,}5} = 2{,}00$, então $P(X_A > 3) = 1 - \Phi(2{,}00) \approx 0{,}0228$;
- B: $z = \frac{3 - 1{,}8}{0{,}9} \approx 1{,}33$, então $P(X_B > 3) = 1 - \Phi(1{,}33) \approx 0{,}0912$.

### O que o código faz
Calcula as duas probabilidades e desenha as duas curvas no mesmo gráfico, com a linha dos 3 s.

In [ ]:
servidores = {"A (atual)": (2.0, 0.5), "B (novo)": (1.8, 0.9)}

x = np.linspace(-1, 5, 500)
for nome, (m, s) in servidores.items():
    z = (3 - m) / s                                    # padroniza o limite de 3 s
    print(f"Servidor {nome}: média {m} s, desvio {s} s -> z = {z:.2f}, P(X > 3) = {1 - norm.cdf(z):.4f}")
    plt.plot(x, norm.pdf(x, m, s), linewidth=2, label=f"servidor {nome}")

plt.axvline(3, color="firebrick", linestyle="--", label="limite de 3 s")
plt.xlabel("Tempo de carregamento (segundos)")
plt.ylabel("Densidade")
plt.title("Média menor nem sempre é melhor")
plt.legend()
plt.show()

**Como ler:** o servidor B é mais rápido **em média** (1,8 s contra 2,0 s), mas tem **4 vezes mais** páginas lentas: cerca de **9,1%** contra **2,3%**. No gráfico, a curva de B é mais baixa e mais "espalhada": como $\sigma$ é maior, sobra mais área depois dos 3 s.

**Conclusão para o time:** olhar só a média engana. A normal tem **dois** parâmetros, e o desvio padrão $\sigma$ pesa tanto quanto a média $\mu$ na hora de decidir.

## Para fechar

Neste notebook, o time de dados usou a **distribuição normal** da Aula 13 para:

- calcular a fração de páginas rápidas e lentas pela **padronização** $Z = (X - \mu)/\sigma$ e pela **tabela** ($\Phi$);
- fazer o **caminho contrário** — achar o tempo $t$ para uma meta de 95% ou 99% (o percentil);
- **comparar dois servidores**, mostrando que o desvio padrão importa tanto quanto a média.

Na **Aula 15**, a normal vai servir de **atalho para a binomial** quando o número de ensaios é grande — por exemplo, para analisar um teste A/B com milhares de visitantes.